# 03 - Statistical Analysis

[Back to the EDA notebook](02_eda.ipynb)

This notebook explains the key statistical ideas we use, then tests two hypotheses that came out of the exploration.

## Setup: reusable functions and loading the data
We define the functions used in this notebook, then load the cleaned data and pick out total monthly receipts.

In [5]:

# Bring in pandas, our tool for working with tables
import pandas as pd
# Bring in scipy's statistics tools
from scipy import stats


def load_clean_data(file_path="../data/processed/hmrc_clean.csv"):
    """Load the cleaned HMRC data with Month as the row label."""
    # Read the file, turning Month back into real dates
    data = pd.read_csv(file_path, parse_dates=["Month"])

    # Use Month as the row label and hand the table back
    return data.set_index("Month")


def describe_series(series):
    """Return the mean, median and standard deviation of a series."""
    # These three numbers summarise the middle and the spread of the data
    return pd.Series({"Mean": series.mean(),
                      "Median": series.median(),
                      "Standard deviation": series.std()}).round(0)


def probability_above(series, threshold):
    """Give the percentage of months where the value is above a threshold."""
    # (series > threshold) is True or False for each month, and the average of those is the share
    return round((series > threshold).mean() * 100, 1)

def seasonal_index(data):
    """Compare each month with the typical level around it (1.0 = a normal month)."""
    # A 12-month average centred on each month smooths out the long-term trend
    typical = data.rolling(12, center=True).mean()

    # Dividing by that average leaves only the seasonal pattern
    return data / typical


def january_test(series):
    """Test whether January values are higher than the other months."""
    # Split the series into January months and all the other months
    january = series[series.index.month == 1]
    other = series[series.index.month != 1]

    # Welch's t-test compares two averages without assuming they vary equally
    # alternative="greater" asks: is January higher?
    result = stats.ttest_ind(january, other, equal_var=False, alternative="greater")

    # Hand back the numbers we need to report
    return pd.Series({"January average": january.mean(),
                      "Other months average": other.mean(),
                      "t statistic": result.statistic,
                      "p-value": result.pvalue})


def correlation_test(first, second):
    """Measure how closely two series move together and test if the link is real."""
    # alternative="greater" asks: is the link positive?
    result = stats.pearsonr(first, second, alternative="greater")

    # Hand back the correlation, its p-value and how many months were used
    return pd.Series({"Correlation (r)": result.statistic,
                      "p-value": result.pvalue,
                      "Months used": len(first)})


def decide(p_value, alpha=0.05):
    """Say whether we reject the null hypothesis at the chosen significance level."""
    # A p-value below alpha means luck is an unlikely explanation
    if p_value < alpha:
        return "Reject the null hypothesis: the evidence supports our hypothesis."
    return "Do not reject the null hypothesis: there is not enough evidence."


# Use our function to load the data
df = load_clean_data()

# Pull out total receipts, the series we'll explain the ideas with
total = df["Total HMRC Receipts"]

# Check the size, which should be (113, 44)
print(df.shape)

(113, 44)




**Action:** Defined three reusable functions (`load_clean_data`, `describe_series` and `probability_above`), then loaded the cleaned HMRC data and picked out total monthly receipts.

**Result:** 113 rows and 44 columns, with `Month` as the row label.

## Key concepts

### 1. Mean, median and standard deviation
- **Mean:** add up all the months and divide by how many there are. It is the "average", but a few very big months can pull it upwards.
- **Median:** line the months up from smallest to biggest and take the one in the middle. It isn't pulled around by extreme months.
- **Standard deviation:** a measure of how spread out the months are. A small number means months are all similar, and a large number means they swing a lot.

Here we use them on total monthly HMRC receipts, in £ million.

In [2]:

# Use our function to get the mean, median and standard deviation of total receipts
describe_series(total)

Mean                  61517.0
Median                59566.0
Standard deviation    16666.0
dtype: float64



**Action:** Calculated the mean, median and standard deviation of total monthly receipts.

**Result:** The mean is about £61.5bn and the median about £59.6bn. The standard deviation is about £16.7bn.

**Note:** The mean sits a little above the median, because a few very large months (the Januaries) pull the average up. The large standard deviation tells us receipts swing a lot from month to month, which matches the seasonal pattern we found in the EDA.

Uses function: `describe_series` (Cell 1)

### 2. Probability
Probability is the chance that something happens, from 0% (never) to 100% (always). With historical data we can work it out simply by counting: out of all the months we have, in what share did the thing happen?

For example: if 4 months out of 113 had receipts above £100bn, the chance of a month above £100bn is 4 out of 113, or about 3.5%.

In [3]:
# What share of months had total receipts above £100bn? (values are in £ million)
print("Above £100bn:", probability_above(total, 100000), "%")

# And above £80bn?
print("Above £80bn:", probability_above(total, 80000), "%")

# Which months were above £100bn?
print(total[total > 100000].index.strftime("%b %Y").tolist())

Above £100bn: 3.5 %
Above £80bn: 10.6 %
['Jan 2023', 'Jan 2024', 'Jan 2025', 'Jan 2026']


**Action:** Counted what share of the 113 months had total receipts above £100bn and above £80bn, and listed the months above £100bn.

**Result:** 3.5% of months (4 out of 113) were above £100bn, and 10.6% were above £80bn. The four months above £100bn were January 2023, 2024, 2025 and 2026.

**Note:** Every month above £100bn is a January, which is a strong clue that January really is different. We'll test that formally in the next section.

Uses function: `probability_above` (Cell 1)

### 3. Hypothesis testing
A hypothesis test asks: "Is what I see in the data real, or could it just be luck?"

- **Null hypothesis (H0):** nothing special is going on. For example, January receipts are no different from other months.
- **Alternative hypothesis (H1):** something is going on. For example, January receipts are higher than other months.
- **p-value:** the chance of seeing a difference this big if the null hypothesis were true. A very small p-value means luck is an unlikely explanation.
- **Significance level (alpha):** the cut-off we choose before testing, usually 0.05 (5%). If the p-value is below it, we reject the null hypothesis. If not, we say there isn't enough evidence.

**t-test:** compares the averages of two groups, for example January months against all other months, and gives a p-value.

### 4. Correlation
Correlation (r) measures how closely two things move together, from -1 to +1. A value near +1 means they rise and fall together, near 0 means no link, and near -1 means one rises while the other falls. The test also gives a p-value, telling us whether the link is likely to be real.

**Important:** correlation shows that two things move together, not that one causes the other.

### What comes next
We will test two hypotheses that came out of the EDA:
1. January receipts are higher than other months (t-test).
2. Income Tax and National Insurance receipts are positively correlated (correlation test).

### Hypothesis 1: January receipts are higher than other months
- **Null hypothesis (H0):** average total receipts in January are the same as in other months.
- **Alternative hypothesis (H1):** average total receipts in January are higher.
- **Test:** Welch's t-test, which compares two averages without assuming they vary equally.
- **Significance level:** 0.05.

In [6]:
# Choose the significance level before looking at any results
alpha = 0.05

# Run the January test on total monthly receipts
january_result = january_test(total)

# Print the results in a readable way
print(f"January average: £{january_result['January average']:,.0f}m")
print(f"Other months average: £{january_result['Other months average']:,.0f}m")
print(f"t statistic: {january_result['t statistic']:.2f}")
print(f"p-value: {january_result['p-value']:.6f}")

# Ask our function for the decision
print(decide(january_result["p-value"], alpha))

January average: £95,490m
Other months average: £58,577m
t statistic: 6.01
p-value: 0.000113
Reject the null hypothesis: the evidence supports our hypothesis.


**Action:** Ran a t-test comparing total receipts in January months with all other months.

**Result:** January months averaged about £95.5bn, against about £58.6bn for the other months, which is roughly 63% higher. The t statistic is 6.01 and the p-value is about 0.0001, well below 0.05, so we reject the null hypothesis. January receipts are significantly higher.

Uses functions: `january_test` and `decide` (Cell 1)

### Check on Hypothesis 1: removing the trend
Receipts grow over the years, which could make later Januaries look unusually high. To check, we repeat the test using the seasonal index, where 1.0 means a typical month for that point in the year. This removes the long-term trend.

In [7]:
# Turn total receipts into the seasonal index, dropping the edge months with no average
season = seasonal_index(total).dropna()

# Run the same January test on the index instead of the raw receipts
season_result = january_test(season)

# Print the results
print(f"January index: {season_result['January average']:.2f}")
print(f"Other months index: {season_result['Other months average']:.2f}")
print(f"t statistic: {season_result['t statistic']:.2f}")
print(f"p-value: {season_result['p-value']:.2e}")

# Ask our function for the decision
print(decide(season_result["p-value"], alpha))

January index: 1.55
Other months index: 0.95
t statistic: 23.24
p-value: 7.49e-13
Reject the null hypothesis: the evidence supports our hypothesis.


**Action:** Repeated the January test on the seasonal index, so that rising receipts over the years can't explain the result.

**Result:** January's index averaged about 1.55, against about 0.95 for the other months. The t statistic is 23.2 and the p-value is far below 0.001, so we reject the null hypothesis again. The January effect is real and not just a trend.

Uses functions: `seasonal_index`, `january_test` and `decide` (Cell 1)

### Hypothesis 2: Income Tax and National Insurance are positively correlated
- **Null hypothesis (H0):** there is no positive correlation between the two.
- **Alternative hypothesis (H1):** they are positively correlated.
- **Test:** Pearson correlation, which gives a score from -1 to +1 and a p-value.
- **Significance level:** 0.05.

In [ ]:

# Pull out the two tax columns we want to compare
income_tax = df["Income Tax"]
nics = df["National Insurance Contributions"]

# Run the correlation test on their monthly receipts
level_result = correlation_test(income_tax, nics)

# Print the results
print(f"Correlation (r): {level_result['Correlation (r)']:.2f}")
print(f"p-value: {level_result['p-value']:.2e}")
print(f"Months used: {int(level_result['Months used'])}")

# Ask our function for the decision
print(decide(level_result["p-value"], alpha)) # Reject H0 if True

Correlation (r): 0.82
p-value: 1.09e-28
Months used: 113
Reject the null hypothesis: the evidence supports our hypothesis.


In [ ]:
# Work out each tax's growth compared with the same month a year earlier
# pct_change(12) compares every month with the one 12 months before it
growth = pd.DataFrame({"Income Tax": income_tax.pct_change(12),
                       "NICs": nics.pct_change(12)}).dropna()

# Run the correlation test on the growth rates
growth_result = correlation_test(growth["Income Tax"], growth["NICs"])

# Print the results
print(f"Correlation (r): {growth_result['Correlation (r)']:.2f}")
print(f"p-value: {growth_result['p-value']:.2e}")
print(f"Months used: {int(growth_result['Months used'])}")

# Ask our function for the decision
print(decide(growth_result["p-value"], alpha)) # Reject H0 if True

Correlation (r): 0.38
p-value: 4.74e-05
Months used: 101
Reject the null hypothesis: the evidence supports our hypothesis.


**Action:** Compared the yearly growth rates of Income Tax and National Insurance, instead of their raw levels.

**Result:** The correlation drops to about 0.38 across 101 months, but the p-value is still far below 0.05, so we reject the null hypothesis. The link is real but moderate.

**Note:** Much of the 0.82 in Cell 6 came from both taxes rising over time. When we look only at growth, the relationship is positive but weaker.

Uses functions: `correlation_test` and `decide` (Cell 1)